# utils.py — Functions by Arnau Soler

This notebook explains, step by step, the functions from `utils.py` that were implemented by Arnau Soler. The aim is to make each function easier to follow, showing both **what it does** and **why it is useful** inside the Quora Question Pairs pipeline.

The notebook starts with simple similarity metrics, then moves towards the full feature extraction process used by the baseline and improved models. Throughout the notebook, the examples are intentionally small so that the behaviour of each function can be understood before seeing how everything is combined.

The topics covered are:

- from-scratch similarity metrics,
- batch feature extraction,
- saving and loading trained objects,
- model evaluation,
- SBERT-based semantic features,
- graph and magic features.


In [3]:
import numpy as np
import pandas as pd
import os, tempfile
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression

import utils
from utils import (
    char_bigram_dice,
    cosine_similarity_tfidf_batch, get_handcrafted_features, get_combined_features,
    save_object, load_object, evaluate_model,
    get_sbert_interaction_features,
    build_freq_dict, build_neighbor_dict, get_graph_features, get_sbert_graph_features,
)


---

## From-scratch similarity metrics

This first block focuses on similarity functions implemented from scratch. They are useful because they let us understand exactly what each metric captures before using it inside the handcrafted feature vector of the baseline model.


### `char_bigram_dice(str1, str2)`

**What it computes:**

$$D(A, B) = \frac{2 \, |A \cap B|}{|A| + |B|}$$

where $A$ and $B$ are **character-bigram multisets** (counts, not sets).
For example `"hello"` -> `["he", "el", "ll", "lo"]`.

**Why character bigrams instead of words?**

- Robust to minor spelling differences: 'colour' vs 'color' share most bigrams.
- Captures partial word overlap: 'run' and 'running' share `ru`, `un`.
- Works well on short strings where word Jaccard is too coarse.

**Why Dice instead of Jaccard?**
Dice weights shared elements more heavily (doubles the numerator), useful when strings have many unique bigrams.

**Implementation:** `Counter` builds multisets; `c1 & c2` takes the element-wise minimum -- the true intersection of counts.


In [4]:
pairs = [
    ("colour",  "color"),
    ("running", "runner"),
    ("python",  "python"),
    ("cat",     "dog"),
    ("a",       ""),
    ("",        ""),
]

for s1, s2 in pairs:
    score = char_bigram_dice(s1, s2)
    print(f"  {s1!r:15} vs {s2!r:15}  ->  Dice = {score:.3f}")


  'colour'        vs 'color'          ->  Dice = 0.667
  'running'       vs 'runner'         ->  Dice = 0.545
  'python'        vs 'python'         ->  Dice = 1.000
  'cat'           vs 'dog'            ->  Dice = 0.000
  'a'             vs ''               ->  Dice = 1.000
  ''              vs ''               ->  Dice = 1.000


In [5]:
# Step-by-step for 'colour' vs 'color'
def bigrams(s):
    s = s.lower()
    return [s[i:i+2] for i in range(len(s)-1)]

bg1 = bigrams("colour")
bg2 = bigrams("color")
c1, c2 = Counter(bg1), Counter(bg2)
intersection = sum((c1 & c2).values())

print(f"bigrams('colour') = {bg1}")
print(f"bigrams('color')  = {bg2}")
print(f"intersection count = {intersection}")
print(f"Dice = 2*{intersection} / ({len(bg1)} + {len(bg2)}) = {2*intersection/(len(bg1)+len(bg2)):.3f}")


bigrams('colour') = ['co', 'ol', 'lo', 'ou', 'ur']
bigrams('color')  = ['co', 'ol', 'lo', 'or']
intersection count = 3
Dice = 2*3 / (5 + 4) = 0.667


---

## Batch feature extraction

This part moves from individual examples to full DataFrame processing. The objective is to show how the different signals are assembled into the matrices used during training.


### `cosine_similarity_tfidf_batch(df, tfidf_vectorizer)`

**What it computes:**

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}$$

applied **pair-wise** between the TF-IDF vectors of `question1` and `question2`, for every row in the DataFrame.

**Why TF-IDF instead of raw counts?**
Common words like 'what' and 'is' inflate raw count overlap. TF-IDF down-weights them and up-weights rare, informative words,
so cosine similarity better reflects semantic closeness.

**Implementation notes:**

- No Python loop -- all operations run on sparse matrices (`X_q1.multiply(X_q2)`).
- Division-by-zero guard: `np.divide(..., where=denom > 0)` returns 0 for empty strings.
- Returns shape `(n_samples, 1)` for direct column stacking.


In [6]:
toy_df = pd.DataFrame({
    "question1": [
        "What is machine learning",
        "How do I install Python",
        "What is machine learning",
        "The sky is blue",
    ],
    "question2": [
        "What is machine learning",
        "How can I install Python 3",
        "How do neural networks work",
        "Why is grass green",
    ],
})

all_texts = list(toy_df["question1"]) + list(toy_df["question2"])
toy_tfidf = TfidfVectorizer(ngram_range=(1,1), min_df=1)
toy_tfidf.fit(all_texts)

sims = cosine_similarity_tfidf_batch(toy_df, toy_tfidf)
toy_df["tfidf_cosine"] = sims.flatten()
display(toy_df)


,question1,question2,tfidf_cosine
0,What is machine learning,What is machine learning,1.000000
1,How do I install Python,How can I install Python 3,0.694698
2,What is machine learning,How do neural networks work,0.000000
3,The sky is blue,Why is grass green,0.095029


### `get_handcrafted_features(df, tfidf_vectorizer)`

Returns a dense `(n_samples, 9)` matrix. Each column is a handcrafted similarity signal:

| Col | Feature | Rationale |
| --- | --- | --- |
| 0 | Jaccard similarity | How much word vocabulary is shared |
| 1 | Length ratio | Duplicate questions tend to have similar lengths |
| 2 | Common-word F1 | Overlap harmonic mean, less sensitive to length than Jaccard |
| 3 | Char-bigram Dice | Catches spelling variants and partial word overlaps |
| 4 | Number mismatch | 'top 5' != 'top 10' -- specific numbers change meaning |
| 5 | Exact number match | Both questions contain the same numbers |
| 6 | Uppercase word match | Shared proper nouns (names, places, brands) |
| 7 | First word match | Same wh-word suggests same question type |
| 8 | TF-IDF cosine | Continuous semantic overlap weighted by word rarity |

**Design choices:**

- Features 4-7 are boolean (0 or 1); features 0-3 and 8 are continuous in [0, 1].
- The per-pair loop handles columns 0-7; column 8 is a single vectorised call to `cosine_similarity_tfidf_batch`.
- `word_pattern = re.compile(r'\\b\\w+\\b')` is pre-compiled outside the loop to avoid repeated compilation overhead.


In [7]:
feats = get_handcrafted_features(toy_df.drop(columns=["tfidf_cosine"]), toy_tfidf)

col_names = [
    "jaccard", "len_ratio", "word_f1", "bigram_dice",
    "num_mismatch", "exact_num_match", "uppercase_match", "first_word_match",
    "tfidf_cosine",
]
feat_df = pd.DataFrame(feats, columns=col_names)
feat_df.insert(0, "question1", toy_df["question1"].values)
feat_df.insert(1, "question2", toy_df["question2"].values)
display(feat_df)


,question1,question2,jaccard,len_ratio,word_f1,bigram_dice,num_mismatch,exact_num_match,uppercase_match,first_word_match,tfidf_cosine
0,What is machine learning,What is machine learning,1.000000,1.000000,1.000000,1.000000,0.0,0.0,0.0,1.0,1.000000
1,How do I install Python,How can I install Python 3,0.571429,0.833333,0.727273,0.808511,1.0,0.0,1.0,1.0,0.694698
2,What is machine learning,How do neural networks work,0.000000,0.800000,0.000000,0.081633,0.0,0.0,0.0,0.0,0.000000
3,The sky is blue,Why is grass green,0.142857,1.000000,0.250000,0.258065,0.0,0.0,0.0,0.0,0.095029


In [8]:
# Quick sanity check: number features
num_df = pd.DataFrame({
    "question1": ["What are the top 10 movies", "How many seasons does it have", "Give me 3 examples"],
    "question2": ["What are the top 5 movies",  "How many seasons does it have", "Give me 3 good examples"],
})
num_feats = get_handcrafted_features(num_df, toy_tfidf)
for i, (_, row) in enumerate(num_df.iterrows()):
    print(f"  '{row.question1}' vs '{row.question2}'")
    print(f"    num_mismatch={num_feats[i,4]:.0f}  exact_num_match={num_feats[i,5]:.0f}\n")


  'What are the top 10 movies' vs 'What are the top 5 movies'
    num_mismatch=0  exact_num_match=0

  'How many seasons does it have' vs 'How many seasons does it have'
    num_mismatch=0  exact_num_match=0

  'Give me 3 examples' vs 'Give me 3 good examples'
    num_mismatch=0  exact_num_match=1



### `get_combined_features(df, count_vectorizer, tfidf_vectorizer)`

Assembly function used by `train_models.ipynb` and `reproduce_results.ipynb` for the baseline model.
It concatenates horizontally:

1. **BoW interaction features** from `get_interaction_features_from_df` (sparse, ~100k+ columns)
2. **Handcrafted features** from `get_handcrafted_features` (dense 9 columns, converted to sparse)

Result: a single `scipy.sparse.csr_matrix` of shape `(n_samples, vocab_size * 2 + 9)`
so `LogisticRegression.fit` only needs one call.


In [9]:
toy_cv = CountVectorizer(ngram_range=(1,1), min_df=1)
toy_cv.fit(all_texts)

X_combined = get_combined_features(toy_df.drop(columns=["tfidf_cosine"]), toy_cv, toy_tfidf)
print(f"Combined feature matrix shape: {X_combined.shape}")
print(f"  Vocabulary size:          {len(toy_cv.vocabulary_)}")
print(f"  BoW interaction block:    {len(toy_cv.vocabulary_)*2} cols (diff + prod)")
print(f"  Handcrafted block:        9 cols")
print(f"  Total:                    {len(toy_cv.vocabulary_)*2 + 9} cols")
print(f"  Sparsity:                 {1 - X_combined.nnz/(X_combined.shape[0]*X_combined.shape[1]):.2%}")


Combined feature matrix shape: (4, 45)
  Vocabulary size:          18
  BoW interaction block:    36 cols (diff + prod)
  Handcrafted block:        9 cols
  Total:                    45 cols
  Sparsity:                 74.44%


---

## Saving and loading trained objects

Two thin wrappers around Python's `pickle` module.
Used throughout `train_models.ipynb` to save vectorizers and classifiers,
and in `reproduce_results.ipynb` to reload them without retraining.

Any Python object can be saved -- sklearn estimators, dicts, numpy arrays, etc.
Files are opened in **binary mode** (`"wb"` / `"rb"`) as required by pickle.


In [10]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "toy_cv.pkl")

    save_object(toy_cv, path)
    print(f"Saved to {path}")

    reloaded_cv = load_object(path)
    print(f"Reloaded. Vocabulary size: {len(reloaded_cv.vocabulary_)}")

    original_out = toy_cv.transform(["machine learning python"]).toarray()
    reloaded_out = reloaded_cv.transform(["machine learning python"]).toarray()
    print(f"Outputs are identical: {np.array_equal(original_out, reloaded_out)}")


Saved to /tmp/tmp_35vyj7r/toy_cv.pkl
Reloaded. Vocabulary size: 18
Outputs are identical: True


---

## Evaluating the model

Returns a dict with five metrics for a given classifier and split:

| Metric | Function | Notes |
| --- | --- | --- |
| `roc_auc` | `roc_auc_score(y, proba)` | Uses predicted probabilities -- threshold-free |
| `precision` | `precision_score(y, pred)` | Fraction of predicted duplicates that are real duplicates |
| `recall` | `recall_score(y, pred)` | Fraction of real duplicates we actually predicted |
| `f1` | `f1_score(y, pred)` | Harmonic mean of precision and recall |
| `accuracy` | `accuracy_score(y, pred)` | Overall correct fraction |

The dict format makes it easy to build a results DataFrame:
collect rows from multiple `evaluate_model` calls and call `pd.DataFrame(rows)`.


In [11]:
np.random.seed(42)
X_demo = np.random.randn(200, 5)
y_demo = (X_demo[:, 0] + X_demo[:, 1] > 0).astype(int)

clf_demo = LogisticRegression(solver="liblinear", random_state=42)
clf_demo.fit(X_demo, y_demo)

result = evaluate_model(clf_demo, X_demo, y_demo, model_name="demo_clf", split_name="train")
print(result)


{'model': 'demo_clf', 'split': 'train', 'roc_auc': np.float64(1.0), 'precision': 1.0, 'recall': 0.9794, 'f1': 0.9896, 'accuracy': 0.99}


In [12]:
# Build a results table like reproduce_results.ipynb does
rows = [
    evaluate_model(clf_demo, X_demo[:100], y_demo[:100], "demo_clf", "train"),
    evaluate_model(clf_demo, X_demo[100:], y_demo[100:], "demo_clf", "test"),
]
display(pd.DataFrame(rows).set_index(["model", "split"]))


roc_auc  precision  recall      f1  accuracy
model    split                                              
demo_clf train      1.0        1.0  0.9811  0.9905      0.99
         test       1.0        1.0  0.9773  0.9885      0.99

---

## SBERT-based semantic features

### Why SBERT?

The baseline model is fundamentally **lexical**: it can only compare questions token by token.
It will never know that 'car' and 'automobile' are the same thing. We need a semantic representation.

I evaluated four transformer-based options:

| Model | Approach | Problem for this task |
| --- | --- | --- |
| RoBERTa (cross-encoder) | Full attention over [CLS] q1 [SEP] q2 | One forward pass per pair -> O(n^2) inference, too slow for 300k pairs |
| DistilBERT | BERT distilled to 60% size | Not fine-tuned for semantic similarity; needs extra pooling |
| **SBERT** (bi-encoder) | Encodes each question independently | O(n) inference; `paraphrase-MiniLM-L6-v2` was fine-tuned on paraphrase detection |
| BART | Seq2seq generative model | Built for summarisation/translation -- wrong tool for similarity |

SBERT with `paraphrase-MiniLM-L6-v2` is the right choice: it was fine-tuned on paraphrase corpora
(the exact Quora task) and encodes 300k sentences in ~1-2 minutes on CPU.

### Feature design

I mirror the BoW interaction design from `get_interaction_features_from_df`:

$$\text{output} = [\, |\mathbf{e}_{q1} - \mathbf{e}_{q2}| \;\|\; \mathbf{e}_{q1} \odot \mathbf{e}_{q2} \,]$$

- **Absolute difference** `|emb_q1 - emb_q2|` (cols 0-383): captures *what is different*.
For a duplicate pair the embeddings are close, so most dimensions will be near 0.
- **Element-wise product** `emb_q1 * emb_q2` (cols 384-767): captures *what is shared*.
Dimensions pointing in the same direction get large positive values.

Together these give the downstream `LogisticRegression` 768 continuous, semantics-aware signals per pair.


In [13]:
# Demonstrate output shape without loading SBERT
# (loading requires `pip install sentence-transformers` -- done once in train_models.ipynb)

np.random.seed(0)
n_pairs = 4
dim = 384
emb_q1 = np.random.randn(n_pairs, dim).astype(np.float32)
emb_q2 = np.random.randn(n_pairs, dim).astype(np.float32)

diff = np.abs(emb_q1 - emb_q2)     # (n, 384)
prod = emb_q1 * emb_q2             # (n, 384)
X_sbert = np.hstack([diff, prod]).astype(np.float32)  # (n, 768)

print(f"emb_q1 shape:  {emb_q1.shape}")
print(f"emb_q2 shape:  {emb_q2.shape}")
print(f"diff shape:    {diff.shape}")
print(f"prod shape:    {prod.shape}")
print(f"X_sbert shape: {X_sbert.shape}  <- fed to LogisticRegression")


emb_q1 shape:  (4, 384)
emb_q2 shape:  (4, 384)
diff shape:    (4, 384)
prod shape:    (4, 384)
X_sbert shape: (4, 768)  <- fed to LogisticRegression


In [14]:
# Intuition: duplicate pair -> small diff, large prod; non-duplicate -> opposite
base = np.random.randn(dim).astype(np.float32)

dup_q1 = base + np.random.randn(dim).astype(np.float32) * 0.05
dup_q2 = base + np.random.randn(dim).astype(np.float32) * 0.05

non_q1 = np.random.randn(dim).astype(np.float32)
non_q2 = np.random.randn(dim).astype(np.float32)

print("Duplicate pair (nearly identical embeddings):")
print(f"  mean |diff| = {np.abs(dup_q1 - dup_q2).mean():.4f}")
print(f"  mean  prod  = {(dup_q1 * dup_q2).mean():.4f}")

print("\nNon-duplicate pair (unrelated embeddings):")
print(f"  mean |diff| = {np.abs(non_q1 - non_q2).mean():.4f}")
print(f"  mean  prod  = {(non_q1 * non_q2).mean():.4f}")


Duplicate pair (nearly identical embeddings):
  mean |diff| = 0.0575
  mean  prod  = 1.0764

Non-duplicate pair (unrelated embeddings):
  mean |diff| = 1.1037
  mean  prod  = 0.0073


---

## Graph and magic features

### Motivation

Every function so far looks at the **text** of a question pair. But the Quora dataset has a hidden structural property: it is a **graph**.

- Every unique question is a **node**.
- Every CSV row `(q1, q2)` is an **undirected edge**.

Because Quora upsampled duplicate pairs when building the dataset, this graph encodes two extremely strong signals
that virtually every top-10 Kaggle solution exploited:

1. **Node degree / frequency:** A question that appears many times is a hub -- almost always a duplicate.
2. **Common neighbours:** If q1 and q2 share even **one** graph neighbour, the probability of them being a duplicate
drops from ~80% to <40% (InData Labs analysis). This is the single most powerful individual feature in the competition.

### No data leakage

The graph dictionaries are **always built from `train_df` only** and applied read-only to val/test.
Unseen questions get `freq=0` and empty neighbour sets, which is the correct conservative prior.


### `build_freq_dict(df)`

Counts how many times each question appears across both `question1` and `question2` columns.
Equivalent to the **node degree** in the question-pair graph.


In [15]:
train_toy = pd.DataFrame({
    "question1": ["How to learn Python", "How to learn Python", "What is AI",  "How to learn Java"],
    "question2": ["How can I learn Python", "Python learning tips", "What is machine learning", "How to learn Python"],
    "is_duplicate": [1, 1, 0, 0],
})

freq = build_freq_dict(train_toy)

print("Question frequencies (node degrees):")
for q, f in sorted(freq.items(), key=lambda x: -x[1]):
    print(f"  {f}x  '{q}'")


Question frequencies (node degrees):
  3x  'How to learn Python'
  1x  'What is AI'
  1x  'How to learn Java'
  1x  'How can I learn Python'
  1x  'Python learning tips'
  1x  'What is machine learning'


`"How to learn Python"` appears 3 times (twice as q1, once as q2) -- it's a hub node,
and indeed all its pairs in this toy set are labelled as duplicates.


### `build_neighbor_dict(df)`

Builds the adjacency list of the question-pair graph.
`neighbor_dict[q]` = set of all questions directly paired with `q`.
The graph is **undirected**: if `(q1, q2)` is a row, both `q2 in neighbors(q1)` and `q1 in neighbors(q2)`.


In [16]:
neighbors = build_neighbor_dict(train_toy)

print("Adjacency list:")
for q, ns in neighbors.items():
    print(f"  '{q}'")
    for n in ns:
        print(f"      -> '{n}'")


Adjacency list:
  'How to learn Python'
      -> 'Python learning tips'
      -> 'How can I learn Python'
      -> 'How to learn Java'
  'How can I learn Python'
      -> 'How to learn Python'
  'Python learning tips'
      -> 'How to learn Python'
  'What is AI'
      -> 'What is machine learning'
  'What is machine learning'
      -> 'What is AI'
  'How to learn Java'
      -> 'How to learn Python'


### `get_graph_features(df, freq_dict, neighbor_dict)`

Returns a `(n_samples, 6)` float32 array:

| Col | Name | Description |
|-----|------|-------------|
| 0 | `q1_freq` | How often q1 appears in the training graph |
| 1 | `q2_freq` | How often q2 appears in the training graph |
| 2 | `freq_min` | `min(q1_freq, q2_freq)` -- both must be frequent |
| 3 | `freq_max` | `max(q1_freq, q2_freq)` -- at least one is frequent |
| 4 | `freq_diff` | `|q1_freq - q2_freq|` -- asymmetry signal |
| 5 | `intersect` | Size of shared neighbour set -- the key magic feature |


In [17]:
graph_feats = get_graph_features(train_toy, freq, neighbors)

gcols = ["q1_freq", "q2_freq", "freq_min", "freq_max", "freq_diff", "intersect"]
gf_df = pd.DataFrame(graph_feats, columns=gcols)
gf_df.insert(0, "q1", train_toy["question1"].values)
gf_df.insert(1, "q2", train_toy["question2"].values)
gf_df.insert(2, "label", train_toy["is_duplicate"].values)
display(gf_df)


,q1,q2,label,q1_freq,q2_freq,freq_min,freq_max,freq_diff,intersect
0,How to learn Python,How can I learn Python,1,3.0,1.0,1.0,3.0,2.0,0.0
1,How to learn Python,Python learning tips,1,3.0,1.0,1.0,3.0,2.0,0.0
2,What is AI,What is machine learning,0,1.0,1.0,1.0,1.0,0.0,0.0
3,How to learn Java,How to learn Python,0,1.0,3.0,1.0,3.0,2.0,0.0


In [18]:
# Unseen questions get freq=0 and intersect=0 -- correct conservative prior
unseen_df = pd.DataFrame({
    "question1": ["A brand new question never seen before"],
    "question2": ["Another completely new question"],
})
unseen_feats = get_graph_features(unseen_df, freq, neighbors)
print("Graph features for a completely unseen pair:")
print(dict(zip(gcols, unseen_feats[0])))


Graph features for a completely unseen pair:
{'q1_freq': np.float32(0.0), 'q2_freq': np.float32(0.0), 'freq_min': np.float32(0.0), 'freq_max': np.float32(0.0), 'freq_diff': np.float32(0.0), 'intersect': np.float32(0.0)}


### `get_sbert_graph_features(df, sbert_model, freq_dict, neighbor_dict, sbert_feat_path=None)`

Assembly function for the **best model** in the pipeline. Concatenates:

- SBERT interaction features: `(n, 768)` -- loaded from a `.npy` file if it exists, otherwise encoded on the fly.
- Graph features: `(n, 6)` -- from `get_graph_features`.

Result: `(n, 774)` float32 matrix fed to `sbert_graph_logistic`.

**Why load from disk?**
SBERT encoding of 300k questions takes ~1-2 minutes. The `.npy` files are saved by `train_models.ipynb`
so `reproduce_results.ipynb` can evaluate in seconds without re-encoding.


In [19]:
# Demonstrate the assembly without loading the actual SBERT model
with tempfile.TemporaryDirectory() as tmp:
    fake_sbert_path = os.path.join(tmp, "fake_sbert.npy")

    # Simulate pre-computed SBERT features (768 dims)
    fake_X_sbert = np.random.randn(len(train_toy), 768).astype(np.float32)
    np.save(fake_sbert_path, fake_X_sbert)

    X_sg = get_sbert_graph_features(
        train_toy,
        sbert_model=None,           # not needed since the .npy exists
        freq_dict=freq,
        neighbor_dict=neighbors,
        sbert_feat_path=fake_sbert_path,
    )

print(f"SBERT block:  (n, 768)")
print(f"Graph block:  (n,   6)")
print(f"Combined:     {X_sg.shape}  <- expected ({len(train_toy)}, 774)")


SBERT block:  (n, 768)
Graph block:  (n,   6)
Combined:     (4, 774)  <- expected (4, 774)
